# Video Generation with ComfyUI

Welcome to this hands-on workshop! [ComfyUI](https://github.com/comfyanonymous/ComfyUI) is a node-based graphical interface designed for diffusion models, enabling users to visually construct AI image/video generation workflows through modular operations. Its modular node degisn, efficiency, compatibility, and workflow advantage make it a perfect choice for media creators to boost productivity.

This workshop guides you through setting up and running ComfyUI on AMD Instinct GPUs using ROCm™ software. Learn how to configure your environment, install the ComfyUI tool, and generate video form text (or text and image).


# ComfyUI setup

To set up the ComfyUI inference environment, follow the steps below.

#### Verify the PyTorch installation

Verify that PyTorch is correctly installed.

**Step 1** Verify PyTorch is installed and can detect the GPU compute device.

In [ ]:
!python3 -c 'import torch' 2> /dev/null && echo 'Success' || echo 'Failure'

The expected result is `Success`.

**Step 2** Confirm the GPU is available.

In [ ]:
!python3 -c 'import torch; print(torch.cuda.is_available())'

The expected result is `True`.

**Step 3** Display the installed GPU device name.

In [ ]:
!python3 -c "import torch; print(f'device name [0]:', torch.cuda.get_device_name(0))"

The expected result should be similar to: `device name [0]: `

# ComfyUI installation

Install ComfyUI from source on the system with the AMD GPU.

In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI.git

Ensure that PyTorch will not be reinstalled with the CUDA version:

In [ ]:
%cd ComfyUI
!sed -i.bak -E '/^(torch|torchaudio|torchvision)([<>=~!0-9.]*)?$/s/^/# /' requirements.txt

Install the dependencies:

In [ ]:
!pip3 install -r requirements.txt

In [ ]:
!mkdir -p user/default/workflows

In [ ]:
!wget https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors -O models/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors

In [ ]:
!wget https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors -O models/checkpoints/ltx-2.3-22b-dev-fp8.safetensors 
!wget https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors -O models/loras/ltx-2.3-22b-distilled-lora-384.safetensors
!wget https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors -O models/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors
!wget https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors -O models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors

## 🚀 Launch the Server

#### Get the Public App URL

This cell prints the public URL of your notebook environment.

ComfyUI will be exposed through this URL, and we will use it to construct the `--comfy-api-base` parameter when starting the server.

In [ ]:
!echo $APP_URL

You should see a link like: `https://amd-ai-academy.com/app/jupyter-launcher-3673/`

#### Start the ComfyUI Server (Background Mode)

This cell starts the ComfyUI server in the background.
- Logs are written to `comfyui.log`
- The process ID (PID) is saved to `comfyui.pid`

Since the server runs in the background, this cell will complete immediately.

In [ ]:
%%bash
APP_BASE=$(echo "$APP_URL" | sed 's|https://[^/]*||')

nohup python3 main.py \
  --listen 0.0.0.0 \
  --port 8000 \
  --comfy-api-base "$APP_BASE" \
  > comfyui.log 2>&1 &

echo $! > comfyui.pid

sleep 2
ls -l comfyui.log

#### Verify the Server is Running

This cell checks whether the ComfyUI server process is running using the stored PID.

If the server started successfully, you should see a `python3 main.py` process.

In [ ]:
!ps -fp $(cat comfyui.pid)

#### View Server Logs

This cell shows the latest logs from the ComfyUI server.

Use this to:
- Confirm the server started successfully
- Debug issues if the server fails to start

In [ ]:
!tail -n 50 comfyui.log

## Open the ComfyUI Interface

After the server starts, open the value of `APP_URL` in your browser.


For example, if: `APP_URL=https://amd-ai-academy.com/app/jupyter-launcher-3673/`


then open: `https://amd-ai-academy.com/app/jupyter-launcher-3673/`

Once you open `APP_URL`, the ComfyUI interface should appear. 

You’ll see:  
- A **node-based canvas** in the main area  
- A **sidebar on the left**, where you can load workflows and start generating  

From the sidebar, select Templates, then choose **Wan 2.2 5B Video Generation** as shown below:

![ComfyUI Templates](https://raw.githubusercontent.com/Vivicai1005/ai_academy_images/main/comfyui_interfaces/comfyui_template.png)

![ComfyUI Workflow](https://raw.githubusercontent.com/Vivicai1005/ai_academy_images/main/comfyui_interfaces/comfyui_workflow.png)

##### ⚠️ Restart Required After Making Changes

If you plan to update models, workflows, or any notebook configurations, you must restart the ComfyUI server for the changes to take effect.

Follow these steps:

1. **Stop the running server**
   ```bash
   !kill $(cat comfyui.pid)
   ```

2. **Apply your changes**
(e.g., update models, edit workflows, or modify notebook settings)

3. **Restart the server**
Re-run the "Start the ComfyUI server" cell